In [1]:
%pip install tenseal stable_baselines3

   ---------------------------------------- 0.0/952.1 kB ? eta -:--:--
   --------------------------------------- 952.1/952.1 kB 13.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
    --------------------------------------- 2.6/110.9 MB 14.5 MB/s eta 0:00:08
   -- ------------------------------------- 5.8/110.9 MB 15.0 MB/s eta 0:00:08
   --- ------------------------------------ 8.9/110.9 MB 15.1 MB/s eta 0:00:07
   ----- ---------------------------------- 13.9/110.9 MB 17.2 MB/s eta 0:00:06
   ------- -------------------------------- 20.4/110.9 MB 20.1 MB/s eta 0:00:05
   --------- ------------------------------ 26.7/110.9 MB 21.7 MB/s eta 0:00:04
   ----------- ---------------------------- 32.8/110.9 MB 22.8 MB/s eta 0:00:04
   -------------- ------------------------- 39.1/110.9 MB 23.6 MB/s eta 0:00:04
   ---------------- ----------------------- 45.4/110.9 MB 24.3 MB/s eta 0:00:03
   ------------------ --------------------- 51.4/110.9 MB 24.


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

: 

In [ ]:
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)

In [ ]:
#!/usr/bin/env python3
"""
FHE Bootstrapping RL Environment - ENHANCED
- Custom progress tracking (steps, episodes, bootstraps)
- No joblib parallel output clutter
- Clean, readable progress display
"""

import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
import pickle
import time
import logging
from typing import Dict, List, Tuple, Optional
from pathlib import Path

# Disable joblib parallel output
import os
os.environ['JOBLIB_START_METHOD'] = 'fork'
os.environ['JOBLIB_PRINT_PROGRESS'] = '0' # Disable progress messages
os.environ['JOBLIB_WARN_IF_DLOPEN_GLIBC'] = '0' # Disable dlopen warnings

# Set up logging to suppress parallel output
logging.getLogger('joblib').setLevel(logging.CRITICAL)
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CUSTOM PROGRESS TRACKER
# ============================================================================

class ProgressTracker:
    """Track and display training progress cleanly"""

    def __init__(self, total_timesteps: int, eval_freq: int):
        self.total_timesteps = total_timesteps
        self.eval_freq = eval_freq
        self.current_step = 0
        self.episode_count = 0
        self.bootstrap_count = 0
        self.episode_rewards = []
        self.start_time = time.time()
        self.last_eval_time = self.start_time

    def update_step(self, steps: int = 1):
        """Update after N steps"""
        self.current_step += steps

    def update_episode(self, reward: float, episode_length: int, bootstraps: int):
        """Update after episode completion"""
        self.episode_count += 1
        self.episode_rewards.append(reward)
        self.bootstrap_count += bootstraps

    def get_progress_bar(self):
        """Get progress bar string"""
        filled = int(40 * self.current_step // self.total_timesteps)
        bar = '█' * filled + '░' * (40 - filled)
        percentage = (self.current_step / self.total_timesteps) * 100
        return f"[{bar}] {percentage:5.1f}%"

    def print_progress(self, force=False):
        """Print clean progress line"""
        elapsed = time.time() - self.start_time
        steps_per_sec = self.current_step / elapsed if elapsed > 0 else 0
        remaining_steps = self.total_timesteps - self.current_step
        eta_seconds = remaining_steps / steps_per_sec if steps_per_sec > 0 else 0

        # Convert to hours:minutes:seconds
        eta_hours = int(eta_seconds // 3600)
        eta_mins = int((eta_seconds % 3600) // 60)
        eta_secs = int(eta_seconds % 60)

        elapsed_hours = int(elapsed // 3600)
        elapsed_mins = int((elapsed % 3600) // 60)
        elapsed_secs = int(elapsed % 60)

        progress_bar = self.get_progress_bar()

        avg_reward = np.mean(self.episode_rewards[-10:]) if self.episode_rewards else 0

        line = (f"\r{progress_bar} | "
                f"Steps: {self.current_step}/{self.total_timesteps} | "
                f"Episodes: {self.episode_count} | "
                f"Bootstraps: {self.bootstrap_count} | "
                f"Avg Reward: {avg_reward:6.2f} | "
                f"Elapsed: {elapsed_hours}:{elapsed_mins:02d}:{elapsed_secs:02d} | "
                f"ETA: {eta_hours}:{eta_mins:02d}:{eta_secs:02d}")

        print(line, end='', flush=True)

    def print_episode_summary(self, ep: int, total_eps: int, reward: float,
                             length: int, bootstraps: int, success: bool):
        """Print episode summary"""
        status = "✅ SUCCESS" if success else "❌ FAILED"
        print(f"\n   Episode {ep}/{total_eps}: Reward={reward:7.2f} | "
              f"Steps={length:2d} | Bootstraps={bootstraps} | {status}")


# ============================================================================
# FHE CIRCUIT SIMULATOR
# ============================================================================

class FHECircuitSimulator:
    """
    Simulates FHE circuit execution with noise accumulation
    Uses trained noise predictor for realistic noise simulation
    """

    def __init__(self, noise_predictor, scaler, feature_names):
        self.noise_predictor = noise_predictor
        self.scaler = scaler
        self.feature_names = feature_names

        # Circuit operations to simulate
        self.operation_types = ['add', 'multiply', 'square', 'scalar_mult']
        self.operation_type_to_code = {
            'add': 0, 'multiply': 1, 'square': 2, 'scalar_mult': 3
        }

    def create_random_circuit(self, complexity='moderate', length=None):
        """
        Create a random FHE circuit

        Args:
            complexity: 'simple' (easier) or 'moderate' (harder)
            length: Number of operations (default: random based on complexity)

        Returns:
            List of operations with metadata
        """
        if length is None:
            if complexity == 'simple':
                length = np.random.randint(5, 10)
            else:
                length = np.random.randint(8, 15)

        circuit = []
        mult_count = 0
        max_mult = 2 if complexity == 'simple' else 3

        for i in range(length):
            # Choose operation type
            if mult_count >= max_mult:
                op_type = np.random.choice(['add', 'scalar_mult'])
            else:
                op_type = np.random.choice(
                    ['add', 'multiply', 'square', 'scalar_mult'],
                    p=[0.5, 0.25, 0.15, 0.1]
                )

            if op_type in ['multiply', 'square']:
                mult_count += 1

            circuit.append({
                'operation_id': i + 1,
                'operation_type': op_type,
                'operation_type_encoded': self.operation_type_to_code[op_type],
                'multiplicative_depth': mult_count
            })

        return circuit, complexity

    def extract_features_for_operation(self, circuit, op_index, current_state):
        """
        Extract features for a single operation in the current state

        Args:
            circuit: Full circuit definition
            op_index: Current operation index
            current_state: Current execution state

        Returns:
            Feature vector matching trained model
        """
        op = circuit[op_index]
        circuit_length = len(circuit)

        # Build feature dictionary
        features = {
            'operation_type_encoded': op['operation_type_encoded'],
            'is_multiplication': 1 if op['operation_type'] in ['multiply', 'square'] else 0,
            'is_addition': 1 if op['operation_type'] == 'add' else 0,
            'is_square': 1 if op['operation_type'] == 'square' else 0,
            'is_scalar_mult': 1 if op['operation_type'] == 'scalar_mult' else 0,
            'num_inputs': 2 if op['operation_type'] in ['add', 'multiply'] else 1,
            'op_input_magnitude': current_state.get('op_magnitude', 1.0),
            'op_output_magnitude': current_state.get('op_magnitude', 1.0),

            'multiplicative_depth': op['multiplicative_depth'],
            'position_in_circuit_norm': (op_index + 1) / circuit_length,
            'op_index_from_end': circuit_length - op_index - 1,
            'operations_since_bootstrap': current_state['ops_since_bootstrap'],
            'time_since_last_bootstrap': current_state.get('time_since_bootstrap', 0.0),
            'bootstrap_count_so_far': current_state['bootstrap_count'],
            'cumulative_operations': op_index + 1,
            'cum_ops_normalized': (op_index + 1) / circuit_length,

            'poly_modulus_degree': 4096 if current_state['complexity'] == 'simple' else 8192,
            'global_scale_log2': 20,
            'complexity_encoded': 0 if current_state['complexity'] == 'simple' else 1,
            'circuit_type_encoded': 0,

            'cumulative_noise': current_state['cumulative_noise'],
            'delta_decryption_error': current_state.get('last_noise_delta', 0.0),
            'delta_cumulative_noise': current_state.get('last_noise_delta', 0.0),

            'rolling_avg_decryption_error_3': np.mean(current_state['noise_history'][-3:]) if len(current_state['noise_history']) > 0 else 0.0,
            'rolling_max_decryption_error_3': np.max(current_state['noise_history'][-3:]) if len(current_state['noise_history']) > 0 else 0.0,
            'rolling_count_mult_3': sum(1 for i in range(max(0, op_index-2), op_index+1) if circuit[i]['operation_type'] in ['multiply', 'square']),
            'rolling_prop_mult_3': sum(1 for i in range(max(0, op_index-2), op_index+1) if circuit[i]['operation_type'] in ['multiply', 'square']) / 3.0,
            'rolling_avg_op_time_3': 0.01,
            'rolling_avg_decryption_error_5': np.mean(current_state['noise_history'][-5:]) if len(current_state['noise_history']) > 0 else 0.0,
            'rolling_max_decryption_error_5': np.max(current_state['noise_history'][-5:]) if len(current_state['noise_history']) > 0 else 0.0,
            'rolling_prop_mult_5': sum(1 for i in range(max(0, op_index-4), op_index+1) if circuit[i]['operation_type'] in ['multiply', 'square']) / 5.0,
            'rolling_avg_decryption_error_10': np.mean(current_state['noise_history'][-10:]) if len(current_state['noise_history']) > 0 else 0.0,
            'rolling_max_decryption_error_10': np.max(current_state['noise_history'][-10:]) if len(current_state['noise_history']) > 0 else 0.0,
            'rolling_prop_mult_10': sum(1 for i in range(max(0, op_index-9), op_index+1) if circuit[i]['operation_type'] in ['multiply', 'square']) / 10.0,

            'prev_op_type_encoded': circuit[op_index-1]['operation_type_encoded'] if op_index > 0 else 0,
            'prev2_op_type_encoded': circuit[op_index-2]['operation_type_encoded'] if op_index > 1 else 0,

            'mult_depth_times_op_time': op['multiplicative_depth'] * 0.01,
            'mult_depth_ratio': op['multiplicative_depth'] / (2 if current_state['complexity'] == 'simple' else 3),

            'circuit_length': circuit_length,
            'start_noise_proxy': current_state.get('start_noise', 0.0)
        }

        # Convert to array in correct order
        feature_array = np.array([features.get(fname, 0.0) for fname in self.feature_names])

        return feature_array.reshape(1, -1)

    def predict_noise(self, features):
        """Predict noise for given features using trained model"""

        # Standardize features
        features_scaled = self.scaler.transform(features)

        # Predict while suppressing stdout/stderr from joblib
        import io, contextlib, sys
        try:
            buf_out = io.StringIO()
            buf_err = io.StringIO()
            with contextlib.redirect_stdout(buf_out), contextlib.redirect_stderr(buf_err):
                noise_prediction = self.noise_predictor.predict(features_scaled)[0]
        except Exception:
            # fallback: call without redirection if something breaks
            noise_prediction = self.noise_predictor.predict(features_scaled)[0]

        # Clip to reasonable range
        noise_prediction = np.clip(noise_prediction, 0.0, 1.0)

        return noise_prediction



# ============================================================================
# RL ENVIRONMENT
# ============================================================================

class FHEBootstrappingEnv(gym.Env):
    """
    Gymnasium environment for FHE bootstrapping decisions

    State: Current operation features + predicted noise + context
    Action: 0 = execute without bootstrap, 1 = bootstrap then execute
    Reward: Based on correctness (no overflow) and efficiency (fewer bootstraps)
    """

    metadata = {'render_modes': ['human']}

    def __init__(self, noise_predictor, scaler, feature_names,
                 noise_threshold=0.15, max_steps=20, progress_tracker=None):
        super().__init__()

        self.simulator = FHECircuitSimulator(noise_predictor, scaler, feature_names)
        self.noise_threshold = noise_threshold
        self.max_steps = max_steps
        self.progress_tracker = progress_tracker

        # Action space: 0 = execute, 1 = bootstrap
        self.action_space = spaces.Discrete(2)

        # State space
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(10,),
            dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        """Reset environment to start new circuit"""
        super().reset(seed=seed)

        # Generate new circuit
        complexity = np.random.choice(['simple', 'moderate'])
        self.circuit, self.complexity = self.simulator.create_random_circuit(complexity)
        self.circuit_length = len(self.circuit)

        # Initialize state
        self.current_step = 0
        self.state = {
            'cumulative_noise': 0.0,
            'ops_since_bootstrap': 0,
            'bootstrap_count': 0,
            'complexity': complexity,
            'noise_history': [],
            'op_magnitude': 1.0,
            'start_noise': 0.0,
            'last_noise_delta': 0.0,
            'time_since_bootstrap': 0.0
        }

        self.done = False
        self.total_reward = 0.0
        self.bootstrap_history = []

        # Get initial observation
        obs = self._get_observation()
        info = self._get_info()

        return obs, info

    def _get_observation(self):
        """Get current state observation"""
        if self.current_step >= self.circuit_length:
            return np.zeros(10, dtype=np.float32)

        # Extract features and predict noise
        features = self.simulator.extract_features_for_operation(
            self.circuit, self.current_step, self.state
        )
        predicted_noise = self.simulator.predict_noise(features)

        # Compact state representation for RL
        obs = np.array([
            predicted_noise,
            self.state['cumulative_noise'],
            self.state['ops_since_bootstrap'] / self.max_steps,
            (self.circuit_length - self.current_step) / self.circuit_length,
            self.circuit[self.current_step]['multiplicative_depth'] / 3.0,
            1.0 if self.complexity == 'moderate' else 0.0,
            self.state['bootstrap_count'] / 5.0,
            1.0 if self.circuit[self.current_step]['operation_type'] in ['multiply', 'square'] else 0.0,
            np.mean(self.state['noise_history'][-3:]) if len(self.state['noise_history']) > 0 else 0.0,
            self.state['cumulative_noise'] / self.noise_threshold,
        ], dtype=np.float32)

        return obs

    def _get_info(self):
        """Get additional info dict"""
        return {
            'circuit_step': self.current_step,
            'circuit_length': self.circuit_length,
            'cumulative_noise': self.state['cumulative_noise'],
            'bootstrap_count': self.state['bootstrap_count']
        }

    def step(self, action):
        """
        Execute one step in the environment

        Args:
            action: 0 = execute, 1 = bootstrap then execute

        Returns:
            observation, reward, terminated, truncated, info
        """
        if self.done:
            return self._get_observation(), 0.0, True, False, self._get_info()

        # Track bootstrap decision
        bootstrap_performed = (action == 1)
        self.bootstrap_history.append(bootstrap_performed)

        # PERFORM BOOTSTRAP if action == 1
        if bootstrap_performed:
            self.state['cumulative_noise'] = 0.0
            self.state['ops_since_bootstrap'] = 0
            self.state['bootstrap_count'] += 1
            self.state['noise_history'] = []
            bootstrap_time_cost = 0.5
        else:
            bootstrap_time_cost = 0.0

        # EXECUTE OPERATION
        features = self.simulator.extract_features_for_operation(
            self.circuit, self.current_step, self.state
        )
        predicted_noise = self.simulator.predict_noise(features)

        # Simulate actual noise with randomness
        actual_noise = predicted_noise * np.random.uniform(0.8, 1.2)

        # Update state
        old_cumulative = self.state['cumulative_noise']
        self.state['cumulative_noise'] += actual_noise
        self.state['noise_history'].append(actual_noise)
        self.state['last_noise_delta'] = actual_noise
        self.state['ops_since_bootstrap'] += 1
        self.state['time_since_bootstrap'] += 0.01 + bootstrap_time_cost

        # Move to next operation
        self.current_step += 1

        # UPDATE PROGRESS TRACKER
        if self.progress_tracker:
            self.progress_tracker.update_step(1)

        # CHECK TERMINATION CONDITIONS
        terminated = False
        truncated = False

        # 1. Noise overflow (FAILURE)
        if self.state['cumulative_noise'] >= self.noise_threshold:
            terminated = True
            failure_penalty = -100.0
            reward = failure_penalty

        # 2. Circuit completed successfully (SUCCESS)
        elif self.current_step >= self.circuit_length:
            terminated = True
            completion_bonus = 10.0
            bootstrap_penalty = -2.0 * self.state['bootstrap_count']
            reward = completion_bonus + bootstrap_penalty

        # 3. Still running
        else:
            reward = 0.1
            if bootstrap_performed:
                reward -= 1.0

        self.done = terminated or truncated
        self.total_reward += reward

        # Get next observation
        obs = self._get_observation()
        info = self._get_info()

        return obs, reward, terminated, truncated, info

    def render(self, mode='human'):
        """Render current state"""
        if mode == 'human':
            print(f"\nStep {self.current_step}/{self.circuit_length}")
            print(f"  Noise: {self.state['cumulative_noise']:.4f} / {self.noise_threshold}")
            print(f"  Bootstraps: {self.state['bootstrap_count']}")
            print(f"  Reward: {self.total_reward:.2f}")


# ============================================================================
# TEST FUNCTION
# ============================================================================

def test_environment():
    """Test the RL environment"""
    print("="*80)
    print("TESTING FHE BOOTSTRAPPING RL ENVIRONMENT")
    print("="*80)

    # Load trained noise predictor
    print("\n📂 Loading noise predictor...")
    with open('/content/drive/MyDrive/models/noise_predictor/randomforestregressor_best_model.pkl', 'rb') as f:
        model_data = pickle.load(f)

    if hasattr(model_data, 'predict'):
        noise_predictor = model_data
        with open('/content/drive/MyDrive/models/noise_predictor/scaler.pkl', 'rb') as f:
            scaler = pickle.load(f)
        with open('/content/drive/MyDrive/models/noise_predictor/feature_names.pkl', 'rb') as f:
            feature_names = pickle.load(f)
    else:
        noise_predictor = model_data['model']
        scaler = model_data['scaler']
        feature_names = model_data['feature_names']

    print("✅ Noise predictor loaded")

    # --- after loading noise_predictor/scaler/feature_names ---
    # force single-threaded execution to avoid joblib parallel progress output
    try:
        if hasattr(noise_predictor, 'n_jobs'):
            noise_predictor.n_jobs = 8
    except Exception:
        # be quiet if the model doesn't support attribute assignment
        pass

    # Create progress tracker
    progress = ProgressTracker(total_timesteps=5000, eval_freq=1000)

    # Create environment
    print("\n🏗️ Creating RL environment...")
    env = FHEBootstrappingEnv(
        noise_predictor,
        scaler,
        feature_names,
        progress_tracker=progress
    )
    print("✅ Environment created")

    # Test random policy
    print("\n🎮 Testing with random policy...")
    obs, info = env.reset()

    total_episodes = 0
    total_steps = 0

    try:
        for ep in range(10):
            obs, info = env.reset()
            total_episodes += 1

            for step in range(20):
                action = env.action_space.sample()
                obs, reward, terminated, truncated, info = env.step(action)
                total_steps += 1
                progress.print_progress()

                if terminated or truncated:
                    success = info.get('cumulative_noise', 1.0) < env.noise_threshold
                    progress.print_episode_summary(
                        ep+1, 10,
                        reward=env.total_reward,
                        length=step+1,
                        bootstraps=env.state['bootstrap_count'],
                        success=success
                    )
                    break

    except KeyboardInterrupt:
        print("\n\n⏹️  Stopped by user")

    print("\n" + "="*80)
    print("✅ TEST COMPLETE!")
    print("="*80)
    print(f"Total Episodes: {total_episodes}")
    print(f"Total Steps: {total_steps}")
    print(f"Average Episode Length: {total_steps/total_episodes:.1f}")

    return env


if __name__ == "__main__":
    test_environment()

## iter 1

In [ ]:
#!/usr/bin/env python3
"""
Complete RL Agent Training System with Multiple Algorithms
- PPO, A2C, DQN, DDPG
- Model checkpointing & resume capability
- Comprehensive algorithm comparison
- Configurable hyperparameters
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
import time
import json
from datetime import datetime

# RL Libraries
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)

# ============================================================================
# ============================================================================
# TUNABLE HYPERPARAMETERS - MODIFY THESE TO IMPROVE MODEL
# ============================================================================
# ============================================================================

class HyperParameters:
    """
    ADJUST THESE VARIABLES TO IMPROVE MODEL PERFORMANCE
    """

    # ========== TRAINING BASICS ==========
    TOTAL_TIMESTEPS = 50_000          # Total training steps (↑ for better, takes longer)
    N_ENVS = 32*2                       # Parallel environments (↑ faster, uses more RAM)
    EVAL_FREQ = 500                 # Evaluate every N steps (↓ more frequent evals)
    N_EVAL_EPISODES = 10             # Episodes per evaluation (↑ more accurate)

    # ========== LEARNING RATE & OPTIMIZATION ==========
    LEARNING_RATE = 3e-4             # Learning rate (↓ slower learning, ↑ faster but unstable)
    BATCH_SIZE = 64                  # Batch size (↑ more stable, slower; ↓ faster, noisier)
    N_STEPS = 512                   # Steps per batch (PPO specific)
    N_EPOCHS = 10                    # Training epochs (↑ trains longer on same data)
    GAMMA = 0.99                     # Discount factor (↑ values future rewards more)
    GAE_LAMBDA = 0.95                # GAE parameter (smooth advantage estimation)

    # ========== PPO SPECIFIC ==========
    PPO_CLIP_RANGE = 0.2             # Policy clip range (↓ conservative, ↑ aggressive)
    PPO_ENT_COEF = 0.0               # Entropy coefficient (↑ encourages exploration)

    # ========== DQN SPECIFIC ==========
    DQN_BUFFER_SIZE = 50_000          # Replay buffer size (↑ remembers more)
    DQN_LEARNING_STARTS = 1_000       # Steps before training starts
    DQN_TARGET_UPDATE_INTERVAL = 100 # Update target network every N steps
    DQN_EXPLORATION_FRACTION = 0.1   # Fraction of total steps to explore
    DQN_EXPLORATION_FINAL_EPS = 0.05 # Final epsilon for exploration

    # ========== ENVIRONMENT ==========
    NOISE_THRESHOLD = 0.15           # Failure threshold for noise
    MAX_CIRCUIT_STEPS = 20           # Max steps per circuit
    CIRCUIT_COMPLEXITY = 'moderate'  # 'simple' or 'moderate'

    # ========== SAVE/CHECKPOINT ==========
    CHECKPOINT_FREQ = 10_000          # Save checkpoint every N steps
    SAVE_DIR = "/content/drive/MyDrive/models/rl_agent"     # Where to save models
    RESUME_FROM = None               # Set to model path to resume, e.g., "models/rl_agent/checkpoints/ppo_100000"


# ============================================================================
# ============================================================================
# ENVIRONMENT SETUP
# ============================================================================
# ============================================================================

class EnvironmentFactory:
    """Create and manage RL environments"""

    def __init__(self, noise_predictor, scaler, feature_names, hp: HyperParameters):
        self.noise_predictor = noise_predictor
        self.scaler = scaler
        self.feature_names = feature_names
        self.hp = hp

    def make_env(self):
        """Factory function for creating environments"""
        def _init():
            env = FHEBootstrappingEnv(
                self.noise_predictor,
                self.scaler,
                self.feature_names,
                noise_threshold=self.hp.NOISE_THRESHOLD,
                max_steps=self.hp.MAX_CIRCUIT_STEPS
            )
            env = Monitor(env)
            return env
        return _init

    def create_train_env(self):
        """Create vectorized training environment"""
        return DummyVecEnv([self.make_env() for _ in range(self.hp.N_ENVS)])

    def create_eval_env(self):
        """Create evaluation environment"""
        return DummyVecEnv([self.make_env()])


# ============================================================================
# ============================================================================
# MODEL TRAINING & COMPARISON
# ============================================================================
# ============================================================================
from stable_baselines3.common.callbacks import BaseCallback

class StopTrainingOnTimesteps(BaseCallback):
    """Stop training as soon as num_timesteps >= target_timesteps."""
    def __init__(self, target_timesteps: int, verbose: int = 0):
        super().__init__(verbose)
        self.target_timesteps = int(target_timesteps)

    def _on_step(self) -> bool:
        # self.num_timesteps is maintained by SB3
        if self.num_timesteps >= self.target_timesteps:
            if self.verbose > 0:
                print(f"\n[Callback] Reached {self.num_timesteps} timesteps; stopping.")
            # Returning False tells SB3 to stop training.
            return False
        return True

class RLAlgorithmTrainer:
    """Train RL algorithms with consistent interface"""

    def __init__(self, algorithm_name, train_env, eval_env, hp: HyperParameters,
                 output_dir: Path):
        self.algorithm_name = algorithm_name
        self.train_env = train_env
        self.eval_env = eval_env
        self.hp = hp
        self.output_dir = output_dir
        self.model = None
        self.training_time = 0
        self.best_mean_reward = -np.inf
        self.callbacks = []


    def setup_callbacks(self):
        """Setup evaluation and checkpoint callbacks"""
        eval_dir = self.output_dir / self.algorithm_name / "eval"
        checkpoint_dir = self.output_dir / self.algorithm_name / "checkpoints"

        eval_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

        # Evaluation callback (keep verbose low)
        eval_callback = EvalCallback(
            self.eval_env,
            best_model_save_path=str(eval_dir / "best"),
            log_path=str(eval_dir / "logs"),
            eval_freq=self.hp.EVAL_FREQ,
            n_eval_episodes=self.hp.N_EVAL_EPISODES,
            deterministic=True,
            render=False,
            verbose=0
        )

        # Checkpoint callback - set verbose=1 so we get confirmation messages
        # save_freq is in environment steps (i.e., timesteps). With vecenvs this
        # refers to the total environment steps across all envs.
        checkpoint_callback = CheckpointCallback(
            save_freq=self.hp.CHECKPOINT_FREQ,
            save_path=str(checkpoint_dir),
            name_prefix=self.algorithm_name.lower(),
            verbose=1
        )

        # Use CallbackList to combine callbacks (robust)
        self.callbacks = CallbackList([eval_callback, checkpoint_callback])

        # If you're using the StopTrainingOnTimesteps callback from previous advice,
        # append it here for strict stopping:
        try:
            # Only append if not already present
            if not any(cb.__class__.__name__ == 'StopTrainingOnTimesteps' for cb in self.callbacks.callbacks):
                from stable_baselines3.common.callbacks import BaseCallback
                # if StopTrainingOnTimesteps exists in your code use it; otherwise skip
                if 'StopTrainingOnTimesteps' in globals():
                    stop_cb = globals()['StopTrainingOnTimesteps'](self.hp.TOTAL_TIMESTEPS, verbose=0)
                    self.callbacks.append(stop_cb)
        except Exception:
            pass

    def create_model(self):
        """Create model based on algorithm"""
        if self.algorithm_name == 'PPO':
            self.model = PPO(
                "MlpPolicy",
                self.train_env,
                learning_rate=self.hp.LEARNING_RATE,
                n_steps=self.hp.N_STEPS,
                batch_size=self.hp.BATCH_SIZE,
                n_epochs=self.hp.N_EPOCHS,
                gamma=self.hp.GAMMA,
                gae_lambda=self.hp.GAE_LAMBDA,
                clip_range=self.hp.PPO_CLIP_RANGE,
                ent_coef=self.hp.PPO_ENT_COEF,
                verbose=0
            )

        elif self.algorithm_name == 'A2C':
            self.model = A2C(
                "MlpPolicy",
                self.train_env,
                learning_rate=self.hp.LEARNING_RATE,
                n_steps=self.hp.N_STEPS,
                gamma=self.hp.GAMMA,
                gae_lambda=self.hp.GAE_LAMBDA,
                verbose=0
            )

        elif self.algorithm_name == 'DQN':
            self.model = DQN(
                "MlpPolicy",
                self.train_env,
                learning_rate=self.hp.LEARNING_RATE,
                buffer_size=self.hp.DQN_BUFFER_SIZE,
                learning_starts=self.hp.DQN_LEARNING_STARTS,
                target_update_interval=self.hp.DQN_TARGET_UPDATE_INTERVAL,
                exploration_fraction=self.hp.DQN_EXPLORATION_FRACTION,
                exploration_initial_eps=1.0,
                exploration_final_eps=self.hp.DQN_EXPLORATION_FINAL_EPS,
                verbose=0
            )

    def train(self, resume_from=None):
        """Train the model"""
        print(f"\n{'='*70}")
        print(f"Training: {self.algorithm_name}")
        print(f"{'='*70}")

        # Load checkpoint if resuming
        if resume_from and Path(resume_from).exists():
            print(f"📂 Resuming from {resume_from}")
            # use class load to preserve the loaded model correctly
            # Note: keep env set to train_env
            self.model = self.model.load(resume_from, env=self.train_env)
        else:
            self.create_model()

        self.setup_callbacks()

        # Diagnostic: show steps per update & checkpoint dir
        try:
            n_steps = getattr(self.model, 'n_steps', None)
            n_envs = getattr(self.train_env, 'num_envs', None)
            if n_steps is not None and n_envs is not None:
                print(f"[Info] model.n_steps={n_steps}, train_env.num_envs={n_envs}, steps_per_update={n_steps * n_envs}")
        except Exception:
            pass

        # Print where checkpoints will be stored
        checkpoint_dir = (self.output_dir / self.algorithm_name / "checkpoints")
        print(f"[Info] Checkpoints will be saved to: {checkpoint_dir}")

        print(f"⏱️ Training for {self.hp.TOTAL_TIMESTEPS:,} steps...")
        start_time = time.time()

        # Ensure we pass the CallbackList (self.callbacks)
        self.model.learn(
            total_timesteps=self.hp.TOTAL_TIMESTEPS,
            callback=self.callbacks,
            progress_bar=True,
            reset_num_timesteps=(resume_from is None)
        )

        self.training_time = time.time() - start_time

        # After training: list contents of the checkpoint directory for verification
        try:
            files = sorted(checkpoint_dir.glob("*"))
            if files:
                print("\n[Info] Saved checkpoint files:")
                for f in files:
                    print("   -", f.name)
            else:
                print("\n[Warning] No checkpoint files found in", checkpoint_dir)
        except Exception as e:
            print("\n[Warning] Could not list checkpoint directory:", e)

        # Save final model
        final_path = self.output_dir / self.algorithm_name / "final_model"
        self.model.save(final_path)

        print(f"✅ Training complete ({self.training_time/60:.1f} min)")

    def evaluate(self, n_episodes=20, deterministic=True):
        """Evaluate trained model"""
        episode_rewards = []
        episode_lengths = []
        episode_bootstraps = []
        episode_successes = []
        episode_failures = []

        for ep in range(n_episodes):
            obs = self.eval_env.reset()
            done = False
            episode_reward = 0
            step_count = 0
            bootstrap_count = 0
            failure = False

            while not done:
                action, _ = self.model.predict(obs, deterministic=deterministic)
                obs, reward, done, info = self.eval_env.step(action)

                episode_reward += reward[0]
                step_count += 1

                if action[0] == 1:
                    bootstrap_count += 1

                if done[0]:
                    final_info = info[0] if isinstance(info, list) else info
                    cum_noise = final_info.get('cumulative_noise', 1.0)
                    if cum_noise >= self.hp.NOISE_THRESHOLD:
                        failure = True
                    break

            episode_rewards.append(episode_reward)
            episode_lengths.append(step_count)
            episode_bootstraps.append(bootstrap_count)
            episode_successes.append(1 if not failure else 0)
            episode_failures.append(1 if failure else 0)

        results = {
            'algorithm': self.algorithm_name,
            'mean_reward': np.mean(episode_rewards),
            'std_reward': np.std(episode_rewards),
            'mean_length': np.mean(episode_lengths),
            'mean_bootstraps': np.mean(episode_bootstraps),
            'success_rate': np.mean(episode_successes),
            'failure_rate': np.mean(episode_failures),
            'training_time': self.training_time,
            'n_episodes': n_episodes
        }

        return results, episode_rewards, episode_lengths, episode_bootstraps


# ============================================================================
# ============================================================================
# MAIN TRAINING PIPELINE
# ============================================================================
# ============================================================================

class MultiAlgorithmTrainingPipeline:
    """Train and compare multiple RL algorithms"""

    def __init__(self, noise_predictor_path, scaler_path, feature_names_path,
                 hp: HyperParameters):
        self.hp = hp
        self.output_dir = Path(hp.SAVE_DIR)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Load noise predictor
        self.load_components(noise_predictor_path, scaler_path, feature_names_path)

        # Results tracking
        self.all_results = []
        self.algorithm_results = {}

    def load_components(self, noise_predictor_path, scaler_path, feature_names_path):
        """Load noise predictor and preprocessing components"""
        print("\n" + "="*70)
        print("Loading Components")
        print("="*70)

        with open(noise_predictor_path, 'rb') as f:
            self.noise_predictor = pickle.load(f)

        with open(scaler_path, 'rb') as f:
            self.scaler = pickle.load(f)

        with open(feature_names_path, 'rb') as f:
            self.feature_names = pickle.load(f)

        print("✅ Components loaded")

    def train_all_algorithms(self):
        """Train all algorithms"""
        algorithms = ['PPO', 'A2C', 'DQN']

        for algo in algorithms:
            # Create environments
            env_factory = EnvironmentFactory(
                self.noise_predictor, self.scaler, self.feature_names, self.hp
            )
            train_env = env_factory.create_train_env()
            eval_env = env_factory.create_eval_env()

            # Create and train
            trainer = RLAlgorithmTrainer(algo, train_env, eval_env, self.hp,
                                        self.output_dir)

            # Resume if specified
            resume_path = None
            if self.hp.RESUME_FROM and algo.lower() in self.hp.RESUME_FROM.lower():
                resume_path = self.hp.RESUME_FROM

            trainer.train(resume_from=resume_path)

            # Evaluate
            results, rewards, lengths, bootstraps = trainer.evaluate(n_episodes=20)
            self.algorithm_results[algo] = {
                'trainer': trainer,
                'results': results,
                'rewards': rewards,
                'lengths': lengths,
                'bootstraps': bootstraps
            }

            self.all_results.append(results)

            # Cleanup
            train_env.close()
            eval_env.close()

    def compare_algorithms(self):
        """Compare all trained algorithms"""
        print("\n" + "="*70)
        print("Algorithm Comparison Results")
        print("="*70)

        results_df = pd.DataFrame(self.all_results)
        results_df = results_df.sort_values('mean_reward', ascending=False)

        print("\n" + results_df.to_string(index=False))

        # Save comparison
        results_df.to_csv(self.output_dir / "algorithm_comparison.csv", index=False)
        print(f"\n💾 Saved to {self.output_dir / 'algorithm_comparison.csv'}")

        return results_df

    def get_best_model(self):
        """Get best performing algorithm"""
        best_algo = max(self.algorithm_results.items(),
                       key=lambda x: x[1]['results']['mean_reward'])

        print("\n" + "="*70)
        print(f"🏆 Best Algorithm: {best_algo[0]}")
        print(f"Mean Reward: {best_algo[1]['results']['mean_reward']:.2f}")
        print(f"Success Rate: {best_algo[1]['results']['success_rate']*100:.1f}%")
        print("="*70)

        return best_algo[0], best_algo[1]['trainer']

    def plot_results(self):
        """Plot performance comparison"""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Mean reward
        algo_names = [r['algorithm'] for r in self.all_results]
        mean_rewards = [r['mean_reward'] for r in self.all_results]
        std_rewards = [r['std_reward'] for r in self.all_results]

        axes[0, 0].bar(algo_names, mean_rewards, yerr=std_rewards, capsize=5)
        axes[0, 0].set_ylabel('Mean Reward')
        axes[0, 0].set_title('Average Reward per Algorithm')
        axes[0, 0].grid(True, alpha=0.3)

        # Success rate
        success_rates = [r['success_rate']*100 for r in self.all_results]
        axes[0, 1].bar(algo_names, success_rates, color='green', alpha=0.7)
        axes[0, 1].set_ylabel('Success Rate (%)')
        axes[0, 1].set_title('Circuit Success Rate')
        axes[0, 1].set_ylim([0, 100])
        axes[0, 1].grid(True, alpha=0.3)

        # Average bootstraps
        mean_bootstraps = [r['mean_bootstraps'] for r in self.all_results]
        axes[1, 0].bar(algo_names, mean_bootstraps, color='orange', alpha=0.7)
        axes[1, 0].set_ylabel('Avg Bootstraps')
        axes[1, 0].set_title('Average Bootstraps per Episode')
        axes[1, 0].grid(True, alpha=0.3)

        # Training time
        train_times = [r['training_time']/60 for r in self.all_results]
        axes[1, 1].bar(algo_names, train_times, color='red', alpha=0.7)
        axes[1, 1].set_ylabel('Time (minutes)')
        axes[1, 1].set_title('Training Time')
        axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(self.output_dir / "algorithm_comparison.png", dpi=150)
        print(f"📊 Plot saved to {self.output_dir / 'algorithm_comparison.png'}")
        plt.close()

    def save_hyperparameters(self):
        """Save hyperparameters used"""
        hp_dict = {k: v for k, v in vars(self.hp).items()
                  if not k.startswith('_')}

        with open(self.output_dir / "hyperparameters.json", 'w') as f:
            json.dump(hp_dict, f, indent=2)

        print(f"💾 Hyperparameters saved")


# ============================================================================
# ============================================================================
# MAIN EXECUTION
# ============================================================================
# ============================================================================

def main():
    """Main execution"""
    print("\n" + "="*70)
    print("🤖 MULTI-ALGORITHM RL TRAINING PIPELINE")
    print("="*70)

    # Initialize hyperparameters
    hp = HyperParameters()

    print(f"\n📋 Configuration:")
    print(f"   Total Steps: {hp.TOTAL_TIMESTEPS:,}")
    print(f"   Batch Size: {hp.BATCH_SIZE}")
    print(f"   Learning Rate: {hp.LEARNING_RATE}")
    print(f"   Eval Freq: {hp.EVAL_FREQ:,}")

    # Initialize pipeline
    pipeline = MultiAlgorithmTrainingPipeline(
        "/content/drive/MyDrive/models/noise_predictor/randomforestregressor_best_model.pkl",
        "/content/drive/MyDrive/models/noise_predictor/scaler.pkl",
        "/content/drive/MyDrive/models/noise_predictor/feature_names.pkl",
        hp
    )

    # Train all algorithms
    pipeline.train_all_algorithms()

    # Compare
    comparison = pipeline.compare_algorithms()

    # Get best model
    best_algo, best_trainer = pipeline.get_best_model()

    # Visualize
    pipeline.plot_results()

    # Save hyperparameters
    pipeline.save_hyperparameters()

    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE!")
    print("="*70)
    print(f"📁 All results in: {hp.SAVE_DIR}")
    print(f"   - algorithm_comparison.csv: Detailed results")
    print(f"   - algorithm_comparison.png: Visualization")
    print(f"   - [Algorithm]/checkpoints/: Saved checkpoints for resuming")
    print(f"   - [Algorithm]/final_model: Final trained model")


if __name__ == "__main__":
    main()


## iter 2

In [ ]:
#!/usr/bin/env python3
"""
Fine-tune RL Models for Better Performance
- Adjust environment difficulty
- Improve reward shaping
- Optimize hyperparameters
- Create challenging scenarios
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
import json
import time

from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)
# ============================================================================
# ENHANCED ENVIRONMENT WITH BETTER REWARD SHAPING
# ============================================================================

# from fhe_rl_environment_enhanced import FHEBootstrappingEnv

class ChallengedFHEBootstrappingEnv(FHEBootstrappingEnv):
    """
    Enhanced environment with:
    - Better reward shaping
    - More realistic noise accumulation
    - Stricter noise threshold
    - Longer circuits
    """

    def __init__(self, noise_predictor, scaler, feature_names,
                 noise_threshold=0.10,  # REDUCED from 0.15 (harder)
                 max_steps=25,           # INCREASED from 20 (longer circuits)
                 noise_scaling=1.5,      # INCREASED noise accumulation
                 progress_tracker=None):

        super().__init__(
            noise_predictor,
            scaler,
            feature_names,
            noise_threshold=noise_threshold,
            max_steps=max_steps,
            progress_tracker=progress_tracker
        )

        self.noise_scaling = noise_scaling

    def step(self, action):
        """Enhanced step with better reward shaping"""

        if self.done:
            return self._get_observation(), 0.0, True, False, self._get_info()

        bootstrap_performed = (action == 1)
        self.bootstrap_history.append(bootstrap_performed)

        # PERFORM BOOTSTRAP
        if bootstrap_performed:
            self.state['cumulative_noise'] = 0.0
            self.state['ops_since_bootstrap'] = 0
            self.state['bootstrap_count'] += 1
            self.state['noise_history'] = []
            bootstrap_time_cost = 0.5
        else:
            bootstrap_time_cost = 0.0

        # EXECUTE OPERATION with SCALED NOISE
        features = self.simulator.extract_features_for_operation(
            self.circuit, self.current_step, self.state
        )
        predicted_noise = self.simulator.predict_noise(features)

        # SCALE NOISE to make environment harder
        actual_noise = (predicted_noise * np.random.uniform(0.9, 1.3)) * self.noise_scaling

        old_cumulative = self.state['cumulative_noise']
        self.state['cumulative_noise'] += actual_noise
        self.state['noise_history'].append(actual_noise)
        self.state['last_noise_delta'] = actual_noise
        self.state['ops_since_bootstrap'] += 1
        self.state['time_since_bootstrap'] += 0.01 + bootstrap_time_cost

        self.current_step += 1

        if self.progress_tracker:
            self.progress_tracker.update_step(1)

        # ======= IMPROVED REWARD SHAPING =======
        terminated = False
        truncated = False
        reward = 0.0

        # 1. PENALTY for letting noise grow (encourage proactive bootstrapping)
        noise_ratio = self.state['cumulative_noise'] / self.noise_threshold
        if noise_ratio > 0.7:
            reward -= 2.0  # Strong penalty when approaching threshold
        elif noise_ratio > 0.5:
            reward -= 0.5  # Moderate penalty
        elif noise_ratio > 0.3:
            reward -= 0.1  # Mild penalty
        else:
            reward += 0.05  # Small reward for keeping noise low

        # 2. PENALTY for bootstrapping (but not as harsh)
        if bootstrap_performed:
            reward -= 0.5  # Cost of bootstrap

        # 3. REWARD for surviving operations
        reward += 0.1  # Small reward per operation

        # 4. REWARD for efficiency (long circuits with few bootstraps)
        if self.current_step > 0:
            reward += 0.01 * (1 - (self.state['bootstrap_count'] / max(1, self.current_step / 5)))

        # CHECK TERMINATION
        if self.state['cumulative_noise'] >= self.noise_threshold:
            terminated = True
            reward -= 10.0  # Harsh penalty for failure

        elif self.current_step >= self.circuit_length:
            terminated = True
            # SUCCESS! Reward scales with few bootstraps
            reward += 10.0  # Base completion reward
            reward += 2.0 * (1 - self.state['bootstrap_count'] / 5)  # Bonus for efficiency

        self.done = terminated or truncated
        self.total_reward += reward

        obs = self._get_observation()
        info = self._get_info()

        return obs, reward, terminated, truncated, info


# ============================================================================
# FINE-TUNING CONFIGURATION
# ============================================================================

class FineTuningConfig:
    """Hyperparameters for fine-tuning"""

    # ===== ENVIRONMENT =====
    NOISE_THRESHOLD = 0.10          # Stricter (was 0.15)
    MAX_STEPS = 25                  # Longer circuits (was 20)
    NOISE_SCALING = 1.5             # Amplify noise (was 1.0)

    # ===== TRAINING =====
    TOTAL_TIMESTEPS = 100000        # More training (was 50000)
    EVAL_FREQ = 5000
    N_EVAL_EPISODES = 20

    # ===== PPO SPECIFIC =====
    PPO_LEARNING_RATE = 1e-4        # Lower (was 3e-4) - more stable
    PPO_BATCH_SIZE = 128            # Larger (was 64) - smoother
    PPO_N_EPOCHS = 20               # More epochs (was 10)
    PPO_N_STEPS = 4096              # More steps per batch (was 2048)
    PPO_GAMMA = 0.99                # High discount (keep)
    PPO_CLIP_RANGE = 0.1            # Conservative (was 0.2)
    PPO_ENT_COEF = 0.01             # Encourage exploration (was 0.0)

    # ===== A2C SPECIFIC =====
    A2C_LEARNING_RATE = 5e-5        # Lower learning rate
    A2C_N_STEPS = 8192
    A2C_GAMMA = 0.99
    A2C_GAE_LAMBDA = 0.95
    A2C_ENT_COEF = 0.01

    # ===== DQN SPECIFIC =====
    DQN_LEARNING_RATE = 1e-4
    DQN_BUFFER_SIZE = 100000        # Larger buffer (was 50000)
    DQN_LEARNING_STARTS = 1000
    DQN_TARGET_UPDATE_INTERVAL = 2000
    DQN_EXPLORATION_FRACTION = 0.2  # Longer exploration (was 0.1)
    DQN_EXPLORATION_FINAL_EPS = 0.02

    # ===== PATHS =====
    BASE_MODEL_DIR = "/content/drive/MyDrive/models/rl_agent"
    FINETUNED_MODEL_DIR = "/content/drive/MyDrive/models/rl_agent_finetuned"


# ============================================================================
# FINE-TUNING PIPELINE
# ============================================================================

class FineTuningPipeline:
    """Fine-tune RL agents with improved settings"""

    def __init__(self, noise_predictor_path, scaler_path, feature_names_path,
                 config: FineTuningConfig):

        self.config = config
        self.output_dir = Path(config.FINETUNED_MODEL_DIR)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Load components
        with open(noise_predictor_path, 'rb') as f:
            model_data = pickle.load(f)
        self.noise_predictor = model_data if hasattr(model_data, 'predict') else model_data['model']

        with open(scaler_path, 'rb') as f:
            self.scaler = pickle.load(f)

        with open(feature_names_path, 'rb') as f:
            self.feature_names = pickle.load(f)

        self.results = []

    def create_environment(self):
        """Create enhanced environment"""
        def make_env():
            env = ChallengedFHEBootstrappingEnv(
                self.noise_predictor,
                self.scaler,
                self.feature_names,
                noise_threshold=self.config.NOISE_THRESHOLD,
                max_steps=self.config.MAX_STEPS,
                noise_scaling=self.config.NOISE_SCALING
            )
            env = Monitor(env)
            return env

        return DummyVecEnv([make_env for _ in range(4)])

    def finetune_ppo(self, base_model_path=None):
        """Fine-tune PPO with improved hyperparameters"""
        print("\n" + "="*80)
        print("🎯 FINE-TUNING PPO")
        print("="*80)

        train_env = self.create_environment()
        eval_env = self.create_environment()

        output_dir = self.output_dir / "PPO"
        output_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n📋 Configuration:")
        print(f"   Learning Rate: {self.config.PPO_LEARNING_RATE}")
        print(f"   Batch Size: {self.config.PPO_BATCH_SIZE}")
        print(f"   N Epochs: {self.config.PPO_N_EPOCHS}")
        print(f"   Clip Range: {self.config.PPO_CLIP_RANGE}")
        print(f"   Entropy Coef: {self.config.PPO_ENT_COEF}")
        print(f"   Environment Noise Scaling: {self.config.NOISE_SCALING}x")
        print(f"   Noise Threshold: {self.config.NOISE_THRESHOLD}")

        # Create or load model
        if base_model_path:
            print(f"\n📂 Loading base model from {base_model_path}")
            model = PPO.load(base_model_path, env=train_env)
        else:
            print(f"\n🔧 Creating new PPO model")
            model = PPO(
                "MlpPolicy",
                train_env,
                learning_rate=self.config.PPO_LEARNING_RATE,
                n_steps=self.config.PPO_N_STEPS,
                batch_size=self.config.PPO_BATCH_SIZE,
                n_epochs=self.config.PPO_N_EPOCHS,
                gamma=self.config.PPO_GAMMA,
                gae_lambda=0.95,
                clip_range=self.config.PPO_CLIP_RANGE,
                ent_coef=self.config.PPO_ENT_COEF,
                verbose=0
            )

        # Callbacks
        eval_callback = EvalCallback(
            eval_env,
            best_model_save_path=str(output_dir / "best"),
            eval_freq=self.config.EVAL_FREQ,
            n_eval_episodes=self.config.N_EVAL_EPISODES,
            deterministic=True,
            verbose=0
        )

        checkpoint_callback = CheckpointCallback(
            save_freq=self.config.EVAL_FREQ,
            save_path=str(output_dir / "checkpoints"),
            name_prefix="ppo"
        )

        # Train
        print(f"\n🚀 Training for {self.config.TOTAL_TIMESTEPS:,} timesteps...")
        start_time = time.time()

        model.learn(
            total_timesteps=self.config.TOTAL_TIMESTEPS,
            callback=[eval_callback, checkpoint_callback],
            progress_bar=True,
            reset_num_timesteps=not bool(base_model_path)
        )

        train_time = time.time() - start_time

        # Save
        model.save(output_dir / "final_model")

        print(f"\n✅ PPO training complete ({train_time/60:.1f} min)")

        train_env.close()
        eval_env.close()

        return model, train_time

    def finetune_a2c(self, base_model_path=None):
        """Fine-tune A2C"""
        print("\n" + "="*80)
        print("🎯 FINE-TUNING A2C")
        print("="*80)

        train_env = self.create_environment()
        eval_env = self.create_environment()

        output_dir = self.output_dir / "A2C"
        output_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n📋 Configuration:")
        print(f"   Learning Rate: {self.config.A2C_LEARNING_RATE}")
        print(f"   N Steps: {self.config.A2C_N_STEPS}")
        print(f"   Entropy Coef: {self.config.A2C_ENT_COEF}")
        print(f"   Environment Noise Scaling: {self.config.NOISE_SCALING}x")

        if base_model_path:
            print(f"\n📂 Loading base model from {base_model_path}")
            model = A2C.load(base_model_path, env=train_env)
        else:
            print(f"\n🔧 Creating new A2C model")
            model = A2C(
                "MlpPolicy",
                train_env,
                learning_rate=self.config.A2C_LEARNING_RATE,
                n_steps=self.config.A2C_N_STEPS,
                gamma=self.config.A2C_GAMMA,
                gae_lambda=self.config.A2C_GAE_LAMBDA,
                ent_coef=self.config.A2C_ENT_COEF,
                verbose=0
            )

        eval_callback = EvalCallback(
            eval_env,
            best_model_save_path=str(output_dir / "best"),
            eval_freq=self.config.EVAL_FREQ,
            n_eval_episodes=self.config.N_EVAL_EPISODES,
            deterministic=True,
            verbose=0
        )

        checkpoint_callback = CheckpointCallback(
            save_freq=self.config.EVAL_FREQ,
            save_path=str(output_dir / "checkpoints"),
            name_prefix="a2c"
        )

        print(f"\n🚀 Training for {self.config.TOTAL_TIMESTEPS:,} timesteps...")
        start_time = time.time()

        model.learn(
            total_timesteps=self.config.TOTAL_TIMESTEPS,
            callback=[eval_callback, checkpoint_callback],
            progress_bar=True,
            reset_num_timesteps=not bool(base_model_path)
        )

        train_time = time.time() - start_time

        model.save(output_dir / "final_model")

        print(f"\n✅ A2C training complete ({train_time/60:.1f} min)")

        train_env.close()
        eval_env.close()

        return model, train_time

    def finetune_dqn(self, base_model_path=None):
        """Fine-tune DQN"""
        print("\n" + "="*80)
        print("🎯 FINE-TUNING DQN")
        print("="*80)

        train_env = self.create_environment()
        eval_env = self.create_environment()

        output_dir = self.output_dir / "DQN"
        output_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n📋 Configuration:")
        print(f"   Learning Rate: {self.config.DQN_LEARNING_RATE}")
        print(f"   Buffer Size: {self.config.DQN_BUFFER_SIZE}")
        print(f"   Exploration Fraction: {self.config.DQN_EXPLORATION_FRACTION}")
        print(f"   Environment Noise Scaling: {self.config.NOISE_SCALING}x")

        if base_model_path:
            print(f"\n📂 Loading base model from {base_model_path}")
            model = DQN.load(base_model_path, env=train_env)
        else:
            print(f"\n🔧 Creating new DQN model")
            model = DQN(
                "MlpPolicy",
                train_env,
                learning_rate=self.config.DQN_LEARNING_RATE,
                buffer_size=self.config.DQN_BUFFER_SIZE,
                learning_starts=self.config.DQN_LEARNING_STARTS,
                target_update_interval=self.config.DQN_TARGET_UPDATE_INTERVAL,
                exploration_fraction=self.config.DQN_EXPLORATION_FRACTION,
                exploration_final_eps=self.config.DQN_EXPLORATION_FINAL_EPS,
                verbose=0
            )

        eval_callback = EvalCallback(
            eval_env,
            best_model_save_path=str(output_dir / "best"),
            eval_freq=self.config.EVAL_FREQ,
            n_eval_episodes=self.config.N_EVAL_EPISODES,
            deterministic=True,
            verbose=0
        )

        checkpoint_callback = CheckpointCallback(
            save_freq=self.config.EVAL_FREQ,
            save_path=str(output_dir / "checkpoints"),
            name_prefix="dqn"
        )

        print(f"\n🚀 Training for {self.config.TOTAL_TIMESTEPS:,} timesteps...")
        start_time = time.time()

        model.learn(
            total_timesteps=self.config.TOTAL_TIMESTEPS,
            callback=[eval_callback, checkpoint_callback],
            progress_bar=True,
            reset_num_timesteps=not bool(base_model_path)
        )

        train_time = time.time() - start_time

        model.save(output_dir / "final_model")

        print(f"\n✅ DQN training complete ({train_time/60:.1f} min)")

        train_env.close()
        eval_env.close()

        return model, train_time

    def run_all_finetuning(self):
        """Fine-tune all models"""
        print("\n" + "="*80)
        print("🚀 RL MODEL FINE-TUNING PIPELINE")
        print("="*80)

        print(f"\n⚙️ ENHANCED ENVIRONMENT SETTINGS:")
        print(f"   Noise Threshold: {self.config.NOISE_THRESHOLD} (was 0.15)")
        print(f"   Max Circuit Length: {self.config.MAX_STEPS} (was 20)")
        print(f"   Noise Scaling: {self.config.NOISE_SCALING}x (amplifies difficulty)")
        print(f"   Total Training Steps: {self.config.TOTAL_TIMESTEPS:,}")

        results = []

        # Fine-tune PPO
        ppo_model, ppo_time = self.finetune_ppo()
        results.append({'Algorithm': 'PPO', 'Training Time (min)': ppo_time/60})

        # Fine-tune A2C
        a2c_model, a2c_time = self.finetune_a2c()
        results.append({'Algorithm': 'A2C', 'Training Time (min)': a2c_time/60})

        # Fine-tune DQN
        dqn_model, dqn_time = self.finetune_dqn()
        results.append({'Algorithm': 'DQN', 'Training Time (min)': dqn_time/60})

        print("\n" + "="*80)
        print("✅ ALL FINE-TUNING COMPLETE!")
        print("="*80)

        results_df = pd.DataFrame(results)
        print("\n" + results_df.to_string(index=False))

        # Save config
        config_path = self.output_dir / "finetuning_config.json"
        config_dict = {k: v for k, v in vars(self.config).items() if not k.startswith('_')}
        with open(config_path, 'w') as f:
            json.dump(config_dict, f, indent=2)

        print(f"\n💾 Configuration saved to {config_path}")
        print(f"📁 Fine-tuned models saved to {self.output_dir}")


# ============================================================================
# MAIN
# ============================================================================

def main():
    """Main fine-tuning execution"""
    config = FineTuningConfig()

    pipeline = FineTuningPipeline(
        "/content/drive/MyDrive/models/noise_predictor/randomforestregressor_best_model.pkl",
        "/content/drive/MyDrive/models/noise_predictor/scaler.pkl",
        "/content/drive/MyDrive/models/noise_predictor/feature_names.pkl",
        config
    )

    pipeline.run_all_finetuning()

    print("\n" + "="*80)
    print("🎯 NEXT STEPS:")
    print("="*80)
    print("1. Evaluate fine-tuned models with: python evaluate_and_compare_models.py")
    print("2. Models are in: models/rl_agent_finetuned/")
    print("3. Compare against baselines to see improvement")


if __name__ == "__main__":
    main()